# Оценка эссе: сравнение промтов и моделей

Что в тетрадке?
- можно прогнать **один промт через одну выбранную модель**;
- можно прогнать **один промт через все модели**;

Поддерживаются:
- модели из OpenRouter
- GigaChat

In [2]:
import os, re, time, uuid
import numpy as np
import pandas as pd
import requests
import time

## Ключи API

In [5]:
gigachat_auth_key=""
openrouter_api_key=""
gigachat_scope = "GIGACHAT_API_PERS"
openrouter_url = "https://openrouter.ai/api/v1/chat/completions"
gigachat_auth_url = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"
gigachat_chat_url = "https://gigachat.devices.sberbank.ru/api/v1/chat/completions"

## Получение access token для GigaChat

Эта ячейка получает access token. Токен действует 30 минут, **не нужно получать токен для каждого запуска!**
Проверка сертификатов отключена.

In [41]:
headers = {
    "Content-Type": "application/x-www-form-urlencoded",
    "Accept": "application/json",
    "RqUID": str(uuid.uuid4()),
    "Authorization": f"Basic {gigachat_auth_key}"}

data = {"scope": gigachat_scope}
response = requests.post(gigachat_auth_url, headers=headers, data=data, timeout=60, verify=False)
response.raise_for_status()
token_data = response.json()
gigachat_access_token = token_data["access_token"]

print("GigaChat access token is ready.")

/opt/anaconda3/lib/python3.12/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ngw.devices.sberbank.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


GigaChat access token is ready.


# Заголовки для запросов:

Токен гигачата действует всего 30  минут, если обновили токен, то эту ячейку тоже нужно запустить. 

In [42]:
headers_openrouter = {"Authorization": f"Bearer {openrouter_api_key}", "Content-Type": "application/json", "HTTP-Referer": "http://localhost", "X-OpenRouter-Title": "essay-prompt-testing"}
headers_gigachat = {"Authorization": f"Bearer {gigachat_access_token}", "Content-Type": "application/json", "Accept": "application/json"}


## Промт 1

In [43]:
prompt_1 = '''
Ты - проверяющий экзамена по английскому языку. 

Сейчас тебе предстоит проверить письменную часть этого экзамена, а именно эссе. 

Твоя задача - оценить эссе по шкале от 0 до 10 по критериям, которые я отправлю, ориентируясь, в том числе, на результаты оценивания других эссе реальным педагогом. Общая оценка складывается из подпунктов: 1) task response, 2) coherence and cohesion, 3) lexical resource and register, 4) register, 5) Grammatical Range and Accuracy.  
Task Response (max 3 points):
0 points: the student does not adequately address any part of the task: there is no introduction, and/or there is no thesis statement in the introduction; the student presents some ideas in the body, which are largely undeveloped or irrelevant; there is no conclusion at all.
1 points: the student responds to the task only in a minimal way or the answer is tangential: the task in the introduction has not been paraphrased, the last sentence does not formulate a thesis statement; the topic sentences in the body paragraphs are difficult to identify and may be repetitive, the position of the author is not clear and is not supported by facts/statistics/examples/illustrations, more than one main idea is discussed in each body paragraph; in the conclusion the thesis statement is not properly paraphrased and does not clarify the position of the author, the conclusion contains some irrelevant ideas which are not discussed in the main body, a concluding signal is not used;
2 points: the student does not fully address all parts of the task: the task in the introduction is only partly paraphrased and ends with a clear thesis statement but it does not fully reflect the main idea of the essay; in the body the student addresses all parts of the task although some parts may be more fully covered than others, each paragraph discusses only one new point and begins with a clear topic sentence, there is 1 argument without any supporting details; in the conclusion some ideas may be missed in summarising, or the thesis statement is not adequately paraphrased, the conclusion does not contain any new/unrelated ideas or information;
3 points: the student fully addresses all parts of the task: the task in the introduction is fully paraphrased and ends with a clear thesis statement that states the main idea/position of the author; in the body the student presents a fully developed position in answer to the question, each paragraph discusses only one new point and begins with a clear topic sentence, each paragraph has 1–2 extended arguments with supporting details (facts, examples, statistics, etc.); the conclusion summarises the main points or paraphrases the thesis statement, begins with a concluding signal, does not contain any new/unrelated ideas or new information;
Coherence and Cohesion (max 2 points):
0 points: the student does not organise information and ideas logically, fails to use linking devices appropriately or repeats them, does not write in paragraphs.
1 points: the student writes a poorly structured essay, uses a limited number of linking devices, does not use paragraphing sufficiently; the ideas are not always logically organised
2 points: the student writes a clearly structured essay, uses a variety of linking devices which connect the ideas appropriately, uses paragraphing sufficiently, the ideas are logically organised;
Lexical Resource and Register (max 2 points):
0 points: – the student only uses basic vocabulary with very limited control of spelling, word formation or word choice; errors are numerous and impede understanding.
1 points: the student uses a sufficient range of vocabulary, but may make 1-2 mistakes in spelling, word formation or word choice;
2 points: the student uses a wide range of vocabulary, including some advanced lexical items; there may be 1-2 inaccuracies;
Grammatical Range and Accuracy (max 2 points):
0 points: – the student uses basic grammar structures or a limited range of structures and/or makes more than 2 grammatical mistakes, some of which impede understanding.
1 points: – the student uses a variety of grammar structures, but may make 2 mistakes;
2 points: the student uses a wide range of grammar structures and may make 1 minor mistake;
Register (max 1 point)
0 points: the style is inappropriate for the task; the register is informal: the student uses contractions, informal colloquialisms and idiomatic expressions, numbering or basic transitions to begin sentences, simple sentences, personal pronouns, imprecise wording.
1 points: the essay is written in an appropriate (academic/neutral) style: the student uses formal vocabulary and grammar structures, academic idiomatic expressions, impersonal language including passive voice where necessary and avoiding personal pronouns; the student may make a few minor stylistic errors. 
Отправляю примеры заданий и три оцененных реальным педагогом эссе. Считай, что оценка и вердикт преподавателя абсолютно верны:
Эссе 1
Задание: 
Some say that because some people are living longer the age at which they retire from work should be raised considerably. To what extent do you agree or disagree?
Ответ студента: 
Some people argue that since life expectancy is increasing, the retirement age should be significantly increased. However, this essay completely disagrees with this statement because younger professionals are increasingly available to replace older workers and retirement is essential for maintaining health and well-being in later life.
Firstly, raising the retirement age is unnecessary due to the growing number of qualified young specialists entering the workforce. In other words, modern education systems are producing more graduates than ever before. As a result, extending the careers of older employees may limit job opportunities for younger generations. For example, recent university graduates often struggle to find employment in competitive fields such as technology and medicine, partly because experienced workers remain in their positions longer. Therefore, keeping retirement at an appropriate age helps maintain a balanced labour market.
Secondly, retirement plays a crucial role in protecting the physical and mental health of older people. The main reason is that ageing is associated with reduced energy levels and increased risk of illness, making full-time work more difficult and stressful. If elderly people continue to work under such conditions, it might negatively affect both their productivity and overall quality of life. For instance, many older adults who retire at the standard age report improvements in their health and emotional well-being, as they have more time to rest and spend time with family. Consequently, forcing people to work longer could harm their health rather than benefit society.
In conclusion, this essay strongly disagrees that the retirement age should be increased, because younger specialists need employment opportunities and retirement is essential for preserving the health of older individuals. Therefore, maintaining the current retirement age is more beneficial for both individuals and society.
Оценка: 10
Комментарий: 
The essay fully addresses the task and presents a clear and well-developed position with relevant, extended, and well-supported ideas, with a clear and logical argument and well-defined paragraphs. Transitions are smooth and effective. A wide range of vocabulary is used accurately and appropriately although repetitively. 
Эссе 2.
Задание: 
We are becoming increasingly dependent on computer technology. It is used in business, crime detection and even to fly planes. What will it be used for in future? Is this dependence on technology a good thing or should we be suspicious of its benefits?
Ответ студента: 
People's lives increasingly depend on computer technology. They help people in various areas of life. I think that it is useful for us to use new technologies and in the future they will work even better in the same areas where they work now.
The main advantage of using technology is that the quality of their work is constantly increasing. Most likely, in the future, computer technology will continue to be used in medicine, construction, but the quality of their work will improve. People should use technology, and not fight it. The fact that many factors depend on technology does not seem scary to me, I think that this shows the importance of technology. For example, at a plant they will set up a program for the manufacture of parts, if it breaks down, it will be a big problem, but without it, it would be difficult for people to work.
On the other hand, there are people who believe that computer technology ruins life. Because a person is smarter than a machine, and will do a better job.
In conclusion, computer technology is useful for a person and should not be feared. It will continue to develop in the areas where it works now, but will become better.
Оценка: 3,5
Комментарий: This essay addresses the task only partially; the format is inappropriate in some places.
Эссе 3.
Задание: 
«Consumerism has become a worldwide phenomenon. What are the advantages and disadvantages of this trend?»
Ответ студента: 
It is a fact that nowadays we live in a hyper-consumer society. People all over the world buy and use more various goods than ever before. This essay will discuss both drawbacks and benefits of this trend.
It is understandable that this way of consumption is substantially beneficial. Positive impact on businesses and national economies all around the globe is probably the first and foremost merit of this trend. Since consumers buy great amounts of goods, proceeds bring a lot of money to different companies. As a result, national budgets are benefiting from it due to various taxes. One more major benefit worth considering is that purchasing goods gives a person a real opportunity to have retail therapy. It means that a wide choice of wares allows you to buy whatever you want. Thanks to it, the stress level goes down.
On the other hand, there are obvious disadvantages to the cult of consumerism. The principle issue with this trend is excessive spending. Nowadays people often buy more goods than is required to meet their needs. Moreover, they frequently purchase even the things they do not need at all just because they have an opportunity to do so. The second downside to be mentioned is that consumerism is unconscious. Customers commonly do not think about the effect their actions make. Excessive buying leads to profound negative implications for our planet. Among them are littering, environmental pollution and so on.
To sum up, while consumerism is beneficial for business and the economy and allows to have retail therapy, it may lead to excessive spending and result in serious environmental problems. Having taken everything into account, I profoundly believe that this trend has more disadvantages since it not only hits our wallets but also threatens the life of future generations. 
Оценка: 6
Комментарий:  you managed to come up with a beautiful essay, good job!) You managed to use some sophisticated vocabulary as well as grammar structures, please keep on doing it!) However, you not always met the structure of the essay, please have a look at it one more time. I do love the arguments that you use but next time please provide more details to your arguments to make them even  more persuasive. Overall, it’s  very good essay) (П), (О)
Ты получишь задание и ответ студента. Правильность грамматических, лексических конструкций и пунктуации ты можешь проверять, используя правила из открытых источников. Аргументы, вступление и заключение, должны четко соответствовать вопросу/теме из задания, которое тебе будет отправлено.
В качестве ответа последовательно выведи оценку по шкале от 0 из 10  и комментарии относительно ошибок, которые были допущены. Писать исправленное эссе не нужно. Не отходи от этой структуры при ответе. (И)
Придерживайся дружелюбного, но делового тона, который обычно характерен для преподавателей. Помни: ты пишешь комментарии для студента, который хочет без лишней воды получить объективную оценку своего умения писать эссе. В комментарии не хвали студента, упоминай только ошибки. 
Результат твоей работы - текстовый ответ. Никакие файлы создавать не нужно. 
'''

## Промт 2

In [44]:
prompt_2 = '''
Привет, оцени студента эссе по критериям по шкале от 0 до 10.
Критерии:
Task Response (max 3 points):
0 points: the student does not adequately address any part of the task: there is no introduction, and/or there is no thesis statement in the introduction; the student presents some ideas in the body, which are largely undeveloped or irrelevant; there is no conclusion at all.
1 points: the student responds to the task only in a minimal way or the answer is tangential: the task in the introduction has not been paraphrased, the last sentence does not formulate a thesis statement; the topic sentences in the body paragraphs are difficult to identify and may be repetitive, the position of the author is not clear and is not supported by facts/statistics/examples/illustrations, more than one main idea is discussed in each body paragraph; in the conclusion the thesis statement is not properly paraphrased and does not clarify the position of the author, the conclusion contains some irrelevant ideas which are not discussed in the main body, a concluding signal is not used;
2 points: the student does not fully address all parts of the task: the task in the introduction is only partly paraphrased and ends with a clear thesis statement but it does not fully reflect the main idea of the essay; in the body the student addresses all parts of the task although some parts may be more fully covered than others, each paragraph discusses only one new point and begins with a clear topic sentence, there is 1 argument without any supporting details; in the conclusion some ideas may be missed in summarising, or the thesis statement is not adequately paraphrased, the conclusion does not contain any new/unrelated ideas or information;
3 points: the student fully addresses all parts of the task: the task in the introduction is fully paraphrased and ends with a clear thesis statement that states the main idea/position of the author; in the body the student presents a fully developed position in answer to the question, each paragraph discusses only one new point and begins with a clear topic sentence, each paragraph has 1–2 extended arguments with supporting details (facts, examples, statistics, etc.); the conclusion summarises the main points or paraphrases the thesis statement, begins with a concluding signal, does not contain any new/unrelated ideas or new information;
Coherence and Cohesion (max 2 points):
0 points: the student does not organise information and ideas logically, fails to use linking devices appropriately or repeats them, does not write in paragraphs.
1 points: the student writes a poorly structured essay, uses a limited number of linking devices, does not use paragraphing sufficiently; the ideas are not always logically organised
2 points: the student writes a clearly structured essay, uses a variety of linking devices which connect the ideas appropriately, uses paragraphing sufficiently, the ideas are logically organised;
Lexical Resource and Register (max 2 points):
0 points: – the student only uses basic vocabulary with very limited control of spelling, word formation or word choice; errors are numerous and impede understanding.
1 points: the student uses a sufficient range of vocabulary, but may make 1-2 mistakes in spelling, word formation or word choice;
2 points: the student uses a wide range of vocabulary, including some advanced lexical items; there may be 1-2 inaccuracies;
Grammatical Range and Accuracy (max 2 points):
0 points: – the student uses basic grammar structures or a limited range of structures and/or makes more than 2 grammatical mistakes, some of which impede understanding.
1 points: – the student uses a variety of grammar structures, but may make 2 mistakes;
2 points: the student uses a wide range of grammar structures and may make 1 minor mistake;
Register (max 1 point)
0 points: the style is inappropriate for the task; the register is informal: the student uses contractions, informal colloquialisms and idiomatic expressions, numbering or basic transitions to begin sentences, simple sentences, personal pronouns, imprecise wording.
1 points: the essay is written in an appropriate (academic/neutral) style: the student uses formal vocabulary and grammar structures, academic idiomatic expressions, impersonal language including passive voice where necessary and avoiding personal pronouns; the student may make a few minor stylistic errors.
Задание: …
Ответ студента: …
'''

## Промт 3

In [45]:
prompt_3 = '''
Выполни следующий алгоритм:

Ознакомься с критериями проверки эссе по английскому языку:

Task Response (max 3 points):
0 points: the student does not adequately address any part of the task: there is no introduction, and/or there is no thesis statement in the introduction; the student presents some ideas in the body, which are largely undeveloped or irrelevant; there is no conclusion at all.
1 points: the student responds to the task only in a minimal way or the answer is tangential: the task in the introduction has not been paraphrased, the last sentence does not formulate a thesis statement; the topic sentences in the body paragraphs are difficult to identify and may be repetitive, the position of the author is not clear and is not supported by facts/statistics/examples/illustrations, more than one main idea is discussed in each body paragraph; in the conclusion the thesis statement is not properly paraphrased and does not clarify the position of the author, the conclusion contains some irrelevant ideas which are not discussed in the main body, a concluding signal is not used;
2 points: the student does not fully address all parts of the task: the task in the introduction is only partly paraphrased and ends with a clear thesis statement but it does not fully reflect the main idea of the essay; in the body the student addresses all parts of the task although some parts may be more fully covered than others, each paragraph discusses only one new point and begins with a clear topic sentence, there is 1 argument without any supporting details; in the conclusion some ideas may be missed in summarising, or the thesis statement is not adequately paraphrased, the conclusion does not contain any new/unrelated ideas or information;
3 points: the student fully addresses all parts of the task: the task in the introduction is fully paraphrased and ends with a clear thesis statement that states the main idea/position of the author; in the body the student presents a fully developed position in answer to the question, each paragraph discusses only one new point and begins with a clear topic sentence, each paragraph 
has 1–2 extended arguments with supporting details (facts, examples, statistics, etc.); the conclusion summarises the main points or paraphrases the thesis statement, begins with a concluding signal, does not contain any new/unrelated ideas or new information;
Coherence and Cohesion (max 2 points):
0 points: the student does not organise information and ideas logically, fails to use linking devices appropriately or repeats them, does not write in paragraphs.
1 points: the student writes a poorly structured essay, uses a limited number of linking devices, does not use paragraphing sufficiently; the ideas are not always logically organised
2 points: the student writes a clearly structured essay, uses a variety of linking devices which connect the ideas appropriately, uses paragraphing sufficiently, the ideas are logically organised;
Lexical Resource and Register (max 2 points):
0 points: – the student only uses basic vocabulary with very limited control of spelling, word formation or word choice; errors are numerous and impede understanding.
1 points: the student uses a sufficient range of vocabulary, but may make 1-2 mistakes in spelling, word formation or word choice;
2 points: the student uses a wide range of vocabulary, including some advanced lexical items; there may be 1-2 inaccuracies;
Grammatical Range and Accuracy (max 2 points):
0 points: – the student uses basic grammar structures or a limited range of structures and/or makes more than 2 grammatical mistakes, some of which impede understanding.
1 points: – the student uses a variety of grammar structures, but may make 2 mistakes;
2 points: the student uses a wide range of grammar structures and may make 1 minor mistake;
Register (max 1 point)
0 points: the style is inappropriate for the task; the register is informal: the student uses contractions, informal colloquialisms and idiomatic expressions, numbering or basic transitions to begin sentences, simple sentences, personal pronouns, imprecise wording.

Изучи примеры трех ответов студентов (написанных эссе по разным заданиям), для которых реальный педагог выставил оценку по шкале от 0 до 10 по критериям, которые я отправил ранее. Оценка и комментарии преподавателя абсолютно верны, не нужно с ними спорить. Вот сами задания, эссе и оценки:
Эссе 1
Задание: 
Some say that because some people are living longer the age at which they retire from work should be raised considerably. To what extent do you agree or disagree?
Ответ студента: 
Some people argue that since life expectancy is increasing, the retirement age should be significantly increased. However, this essay completely disagrees with this statement because younger professionals are increasingly available to replace older workers and retirement is essential for maintaining health and well-being in later life.
Firstly, raising the retirement age is unnecessary due to the growing number of qualified young specialists entering the workforce. In other words, modern education systems are producing more graduates than ever before. As a result, extending the careers of older employees may limit job opportunities for younger generations. For example, recent university graduates often struggle to find employment in competitive fields such as technology and medicine, partly because experienced workers remain in their positions longer. Therefore, keeping retirement at an appropriate age helps maintain a balanced labour market.
Secondly, retirement plays a crucial role in protecting the physical and mental health of older people. The main reason is that ageing is associated with reduced energy levels and increased risk of illness, making full-time work more difficult and stressful. If elderly people continue to work under such conditions, it might negatively affect both their productivity and overall quality of life. For instance, many older adults who retire at the standard age report improvements in their health and emotional well-being, as they have more time to rest and spend time with family. Consequently, forcing people to work longer could harm their health rather than benefit society.
In conclusion, this essay strongly disagrees that the retirement age should be increased, because younger specialists need employment opportunities and retirement is essential for preserving the health of older individuals. Therefore, maintaining the current retirement age is more beneficial for both individuals and society.
Оценка: 10
Комментарий: 
The essay fully addresses the task and presents a clear and well-developed position with relevant, extended, and well-supported ideas, with a clear and logical argument and well-defined paragraphs. Transitions are smooth and effective. A wide range of vocabulary is used accurately and appropriately although repetitively. 
Эссе 2.
Задание: 
We are becoming increasingly dependent on computer technology. It is used in business, crime detection and even to fly planes. What will it be used for in future? Is this dependence on technology a good thing or should we be suspicious of its benefits?
Ответ студента: 
People's lives increasingly depend on computer technology. They help people in various areas of life. I think that it is useful for us to use new technologies and in the future they will work even better in the same areas where they work now.
The main advantage of using technology is that the quality of their work is constantly increasing. Most likely, in the future, computer technology will continue to be used in medicine, construction, but the quality of their work will improve. People should use technology, and not fight it. The fact that many factors depend on technology does not seem scary to me, I think that this shows the importance of technology. For example, at a plant they will set up a program for the manufacture of parts, if it breaks down, it will be a big problem, but without it, it would be difficult for people to work.
On the other hand, there are people who believe that computer technology ruins life. Because a person is smarter than a machine, and will do a better job.
In conclusion, computer technology is useful for a person and should not be feared. It will continue to develop in the areas where it works now, but will become better.
Оценка: 3,5
Комментарий: This essay addresses the task only partially; the format is inappropriate in some places.
Эссе 3.
Задание: 
«Consumerism has become a worldwide phenomenon. What are the advantages and disadvantages of this trend?»
Ответ студента: 
It is a fact that nowadays we live in a hyper-consumer society. People all over the world buy and use more various goods than ever before. This essay will discuss both drawbacks and benefits of this trend.
It is understandable that this way of consumption is substantially beneficial. Positive impact on businesses and national economies all around the globe is probably the first and foremost merit of this trend. Since consumers buy great amounts of goods, proceeds bring a lot of money to different companies. As a result, national budgets are benefiting from it due to various taxes. One more major benefit worth considering is that purchasing goods gives a person a real opportunity to have retail therapy. It means that a wide choice of wares allows you to buy whatever you want. Thanks to it, the stress level goes down.
On the other hand, there are obvious disadvantages to the cult of consumerism. The principle issue with this trend is excessive spending. Nowadays people often buy more goods than is required to meet their needs. Moreover, they frequently purchase even the things they do not need at all just because they have an opportunity to do so. The second downside to be mentioned is that consumerism is unconscious. Customers commonly do not think about the effect their actions make. Excessive buying leads to profound negative implications for our planet. Among them are littering, environmental pollution and so on.
To sum up, while consumerism is beneficial for business and the economy and allows to have retail therapy, it may lead to excessive spending and result in serious environmental problems. Having taken everything into account, I profoundly believe that this trend has more disadvantages since it not only hits our wallets but also threatens the life of future generations. 
Оценка: 6
Комментарий:  you managed to come up with a beautiful essay, good job!) You managed to use some sophisticated vocabulary as well as grammar structures, please keep on doing it!) However, you not always met the structure of the essay, please have a look at it one more time. I do love the arguments that you use but next time please provide more details to your arguments to make them even  more persuasive. Overall, it’s  very good essay) 
Теперь ты выполняешь роль педагога, который оценивает эссе того же формата, что я тебе отправлял, и по тем же критериям. Ты получишь на вход задание и ответ студента. Твоя задача - поставить оценку по шкале от 0 до 10 и дать комментарий в том же формате, что я отправлял. Примерная длина комментария та же.

Задание: …

Эссе:...

'''

## Промт 4

In [46]:
prompt_4 = '''
Task Response (max 3 points):
0 points: the student does not adequately address any part of the task: there is no introduction, and/or there is no thesis statement in the introduction; the student presents some ideas in the body, which are largely undeveloped or irrelevant; there is no conclusion at all.
1 points: the student responds to the task only in a minimal way or the answer is tangential: the task in the introduction has not been paraphrased, the last sentence does not formulate a thesis statement; the topic sentences in the body paragraphs are difficult to identify and may be repetitive, the position of the author is not clear and is not supported by facts/statistics/examples/illustrations, more than one main idea is discussed in each body paragraph; in the conclusion the thesis statement is not properly paraphrased and does not clarify the position of the author, the conclusion contains some irrelevant ideas which are not discussed in the main body, a concluding signal is not used;
2 points: the student does not fully address all parts of the task: the task in the introduction is only partly paraphrased and ends with a clear thesis statement but it does not fully reflect the main idea of the essay; in the body the student addresses all parts of the task although some parts may be more fully covered than others, 
each paragraph discusses only one new point and begins with a clear topic sentence, there is 1 argument without any supporting details; in the conclusion some ideas may be missed in summarising, or the thesis statement is not adequately paraphrased, the conclusion does not contain any new/unrelated ideas or information;
3 points: the student fully addresses all parts of the task: the task in the introduction is fully paraphrased and ends with a clear thesis statement that states the main idea/position of the author; in the body the student presents a fully developed position in answer to the question, each paragraph discusses only one new point and begins with a clear topic sentence, each paragraph has 1–2 extended arguments with supporting details (facts, examples, statistics, etc.); the conclusion summarises the main points or paraphrases the thesis statement, begins with a concluding signal, does not contain any new/unrelated ideas or new information;
Coherence and Cohesion (max 2 points):
0 points: the student does not organise information and ideas logically, fails to use linking devices appropriately or repeats them, does not write in paragraphs.
1 points: the student writes a poorly structured essay, uses a limited number of linking devices, does not use paragraphing sufficiently; the ideas are not always logically organised
2 points: the student writes a clearly structured essay, uses a variety of linking devices which connect the ideas appropriately, uses paragraphing sufficiently, the ideas are logically organised;
Lexical Resource and Register (max 2 points):
0 points: – the student only uses basic vocabulary with very limited control of spelling, word formation or word choice; errors are numerous and impede understanding.
1 points: the student uses a sufficient range of vocabulary, but may make 1-2 mistakes in spelling, word formation or word choice;
2 points: the student uses a wide range of vocabulary, including some advanced lexical items; there may be 1-2 inaccuracies;
Grammatical Range and Accuracy (max 2 points):
0 points: – the student uses basic grammar structures or a limited range of structures and/or makes more than 2 grammatical mistakes, some of which impede understanding.
1 points: – the student uses a variety of grammar structures, but may make 2 mistakes;
2 points: the student uses a wide range of grammar structures and may make 1 minor mistake;
Register (max 1 point)
0 points: the style is inappropriate for the task; the register is informal: the student uses contractions, informal colloquialisms and idiomatic expressions, numbering or basic transitions to begin sentences, simple sentences, personal pronouns, imprecise wording.
1 points: the essay is written in an appropriate (academic/neutral) style: the student uses formal vocabulary and grammar structures, academic idiomatic expressions, impersonal language including passive voice where necessary and avoiding personal pronouns; the student may make a few minor stylistic errors.
Задание: …

Эссе:...
'''

## Промт 5

In [47]:
prompt_5 = '''
Твоя задача - проверить эссе ученика по английскому языку, выставить оценку по шкале от 0 до 10 и написать комментарий. Проблема в том, что у меня нет четкого списка критериев, знаю лишь, что должны соблюдаться:
1) Task Response
2) Coherence and Cohesion
3) Lexical Resource and Register
4) Grammatical Range and Accuracy
5) Register
Также отправляю примеры эссе других учеников, оценку и комментарии от преподавателя. Считай оценку преподавателя и комментарии абсолютно верными.
Эссе 1
Задание: 
Some say that because some people are living longer the age at which they retire from work should be raised considerably. To what extent do you agree or disagree?
Ответ студента: 
Some people argue that since life expectancy is increasing, the retirement age should be significantly increased. However, this essay completely disagrees with this statement because younger professionals are increasingly available to replace older workers and retirement is essential for maintaining health and well-being in later life.
Firstly, raising the retirement age is unnecessary due to the growing number of qualified young specialists entering the workforce. In other words, modern education systems are producing more graduates than ever before. As a result, extending the careers of older employees may limit job opportunities for younger generations. For example, recent university graduates often struggle to find employment in competitive fields such as technology and medicine, partly because experienced workers remain in their positions longer. Therefore, keeping retirement at an appropriate age helps maintain a balanced labour market.
Secondly, retirement plays a crucial role in protecting the physical and mental health of older people. The main reason is that ageing is associated with reduced energy levels and increased risk of illness, making full-time work more difficult and stressful. If elderly people continue to work under such conditions, it might negatively affect both their productivity and overall quality of life. For instance, many older adults who retire at the standard age report improvements in their health and emotional well-being, as they have more time to rest and spend time with family. Consequently, forcing people to work longer could harm their health rather than benefit society.
In conclusion, this essay strongly disagrees that the retirement age should be increased, because younger specialists need employment opportunities and retirement is essential for preserving the health of older individuals. Therefore, maintaining the current retirement age is more beneficial for both individuals and society.
Оценка: 10
Комментарий: 
The essay fully addresses the task and presents a clear and well-developed position with relevant, extended, and well-supported ideas, with a clear and logical argument and well-defined paragraphs. Transitions are smooth and effective. A wide range of vocabulary is used accurately and appropriately although repetitively. 
Эссе 2.
Задание: 
We are becoming increasingly dependent on computer technology. It is used in business, crime detection and even to fly planes. What will it be used for in future? Is this dependence on technology a good thing or should we be suspicious of its benefits?
Ответ студента: 
People's lives increasingly depend on computer technology. They help people in various areas of life. I think that it is useful for us to use new technologies and in the future they will work even better in the same areas where they work now.
The main advantage of using technology is that the quality of their work is constantly increasing. Most likely, in the future, computer technology will continue to be used in medicine, construction, but the quality of their work will improve. People should use technology, and not fight it. The fact that many factors depend on technology does not seem scary to me, I think that this shows the importance of technology. For example, at a plant they will set up a program for the manufacture of parts, if it breaks down, it will be a big problem, but without it, it would be difficult for people to work.
On the other hand, there are people who believe that computer technology ruins life. Because a person is smarter than a machine, and will do a better job.
In conclusion, computer technology is useful for a person and should not be feared. It will continue to develop in the areas where it works now, but will become better.
Оценка: 3,5
Комментарий: This essay addresses the task only partially; the format is inappropriate in some places.
Эссе 3.
Задание: 
«Consumerism has become a worldwide phenomenon. What are the advantages and disadvantages of this trend?»
Ответ студента: 
It is a fact that nowadays we live in a hyper-consumer society. People all over the world buy and use more various goods than ever before. This essay will discuss both drawbacks and benefits of this trend.
It is understandable that this way of consumption is substantially beneficial. Positive impact on businesses and national economies all around the globe is probably the first and foremost merit of this trend. Since consumers buy great amounts of goods, proceeds bring a lot of money to different companies. As a result, national budgets are benefiting from it due to various taxes. One more major benefit worth considering is that purchasing goods gives a person a real opportunity to have retail therapy. It means that a wide choice of wares allows you to buy whatever you want. Thanks to it, the stress level goes down.
On the other hand, there are obvious disadvantages to the cult of consumerism. The principle issue with this trend is excessive spending. Nowadays people often buy more goods than is required to meet their needs. Moreover, they frequently purchase even the things they do not need at all just because they have an opportunity to do so. The second downside to be mentioned is that consumerism is unconscious. Customers commonly do not think about the effect their actions make. Excessive buying leads to profound negative implications for our planet. Among them are littering, environmental pollution and so on.
To sum up, while consumerism is beneficial for business and the economy and allows to have retail therapy, it may lead to excessive spending and result in serious environmental problems. Having taken everything into account, I profoundly believe that this trend has more disadvantages since it not only hits our wallets but also threatens the life of future generations. 
Оценка: 6
Комментарий:  you managed to come up with a beautiful essay, good job!) You managed to use some sophisticated vocabulary as well as grammar structures, please keep on doing it!) However, you not always met the structure of the essay, please have a look at it one more time. I do love the arguments that you use but next time please provide more details to your arguments to make them even  more persuasive. Overall, it’s  very good essay)
Теперь отправляю задание и ответ студента на него. Выставь ему оценку и наши комментарий:
Задание: …
Ответ студента: …

'''

# Теперь посчитаем метрики

In [4]:
score_patterns = [
    r'overall\s*score\s*[:\-]?\s*\**\s*(\d+(?:\.\d+)?)\s*/?\s*(?:10)?\**',
    r'final\s*score\s*[:\-]?\s*\**\s*(\d+(?:\.\d+)?)\s*/?\s*(?:10)?\**',
    r'итог(?:овая)?\s*оценка\s*[:\-]?\s*\**\s*(\d+(?:\.\d+)?)\s*/?\s*(?:10)?\**',
    r'оценка\s*[:\-]?\s*\**\s*(\d+(?:\.\d+)?)\s*/?\s*(?:10)?\**',
    r'\bscore\b\s*[:\-]?\s*\**\s*(\d+(?:\.\d+)?)\s*/?\s*(?:10)?\**',
    r'(\d+(?:\.\d+)?)\s*/\s*10',
]

def extract_overall_score(text):
    text = "" if pd.isna(text) else str(text).replace(",", ".").strip()
    values = [float(match.group(1)) for pattern in score_patterns for match in [re.search(pattern, text, flags=re.I)] if match]
    valid_values = [value for value in values if 0 <= value <= 10]
    return valid_values[0] if valid_values else np.nan

def extract_comment(text):
    return str(text).strip().split("\n", 1)[-1].strip()

def build_user_message(essay_text, task_text):
    return f"Задание:\n{task_text}\n\nОтвет студента:\n{essay_text}"

def clean_arrays(y_true, y_pred):
    y_true = pd.to_numeric(pd.Series(y_true), errors="coerce").to_numpy(dtype=float)
    y_pred = pd.to_numeric(pd.Series(y_pred), errors="coerce").to_numpy(dtype=float)
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    return y_true[mask], y_pred[mask]

def mae(y_true, y_pred):
    y_true, y_pred = clean_arrays(y_true, y_pred)
    return np.abs(y_true - y_pred).mean() if len(y_true) else np.nan

def rmse(y_true, y_pred):
    y_true, y_pred = clean_arrays(y_true, y_pred)
    return np.sqrt(((y_true - y_pred) ** 2).mean()) if len(y_true) else np.nan

def mape(y_true, y_pred):
    y_true, y_pred = clean_arrays(y_true, y_pred)
    mask = y_true != 0
    return np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]).mean() * 100 if mask.sum() else np.nan

def smape(y_true, y_pred):
    y_true, y_pred = clean_arrays(y_true, y_pred)
    den = np.abs(y_true) + np.abs(y_pred)
    mask = den != 0
    return np.mean(2 * np.abs(y_true[mask] - y_pred[mask]) / den[mask]) * 100 if mask.sum() else np.nan

def bias(y_true, y_pred):
    y_true, y_pred = clean_arrays(y_true, y_pred)
    return (y_pred - y_true).mean() if len(y_true) else np.nan

def within_tol(y_true, y_pred, tol=1):
    y_true, y_pred = clean_arrays(y_true, y_pred)
    return (np.abs(y_true - y_pred) <= tol).mean() if len(y_true) else np.nan

def spearman_corr(y_true, y_pred):
    y_true, y_pred = clean_arrays(y_true, y_pred)
    return pd.Series(y_true).corr(pd.Series(y_pred), method="spearman") if len(y_true) > 1 else np.nan

def parse_rate(parse_ok):
    return pd.Series(parse_ok).astype(float).mean() if len(parse_ok) else np.nan

def abs_bias(y_true, y_pred):
    y_true, y_pred = clean_arrays(y_true, y_pred)
    return abs((y_pred - y_true).mean()) if len(y_true) else np.nan


def tail_ratio(y_true, y_pred):
    y_true, y_pred = clean_arrays(y_true, y_pred)
    if not len(y_true):
        return np.nan

    mae_value = np.abs(y_true - y_pred).mean()
    rmse_value = np.sqrt(((y_true - y_pred) ** 2).mean())

    if mae_value == 0:
        return 1.0 if rmse_value == 0 else np.inf

    return rmse_value / mae_value


def quadratic_weighted_kappa(y_true, y_pred, min_rating=0, max_rating=10):
    """
    QWK для ordinal-оценок. метрика ожидает дискретные классы, поэтому предсказания округляются.
    """
    y_true, y_pred = clean_arrays(y_true, y_pred)

    if len(y_true) < 2:
        return np.nan

    y_true = np.rint(y_true).astype(int)
    y_pred = np.rint(y_pred).astype(int)

    y_true = np.clip(y_true, min_rating, max_rating)
    y_pred = np.clip(y_pred, min_rating, max_rating)

    ratings = np.arange(min_rating, max_rating + 1)
    n_ratings = len(ratings)

    observed = np.zeros((n_ratings, n_ratings), dtype=float)

    for true_score, pred_score in zip(y_true, y_pred):
        observed[true_score - min_rating, pred_score - min_rating] += 1

    true_hist = observed.sum(axis=1)
    pred_hist = observed.sum(axis=0)

    expected = np.outer(true_hist, pred_hist) / observed.sum()

    weights = np.zeros((n_ratings, n_ratings), dtype=float)
    for i in range(n_ratings):
        for j in range(n_ratings):
            weights[i, j] = ((i - j) ** 2) / ((n_ratings - 1) ** 2)

    observed_weighted = (weights * observed).sum()
    expected_weighted = (weights * expected).sum()

    if expected_weighted == 0:
        return np.nan

    return 1 - observed_weighted / expected_weighted

def all_metrics(y_true, y_pred, parse_ok=None):
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "tail_ratio": tail_ratio(y_true, y_pred),

        "MAPE_%": mape(y_true, y_pred),
        "sMAPE_%": smape(y_true, y_pred),

        "Bias": bias(y_true, y_pred),
        "abs_Bias": abs_bias(y_true, y_pred),

        "Within_1": within_tol(y_true, y_pred, 1),
        "Within_2": within_tol(y_true, y_pred, 2),

        "Spearman": spearman_corr(y_true, y_pred),
        "QWK": quadratic_weighted_kappa(y_true, y_pred),

        "Parse_rate": parse_rate(parse_ok) if parse_ok is not None else np.nan
    }

In [86]:
gigachat_models = [
    "GigaChat-2-Max",
    "GigaChat-2",
]

openrouter_models = [
    "openai/gpt-oss-120b:free",
    "tencent/hy3-preview:free",
    "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    "inclusionai/ling-2.6-1t:free",
]

all_models = openrouter_models + gigachat_models
temperature = 0.0
max_tokens = 1500
prompt_versions = [("prompt_1", prompt_1), ("prompt_2", prompt_2), ("prompt_3", prompt_3), ("prompt_4", prompt_4), ("prompt_5", prompt_5)]


In [87]:
# ВЫБЕРИТЕ ОДНУ МОДЕЛЬ И ЗАПУСКАЙТЕ ЯЧЕЙКУ
model_name = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"

In [88]:
def get_gigachat_token():
    auth_headers = {"Content-Type": "application/x-www-form-urlencoded", "Accept": "application/json", "RqUID": str(uuid.uuid4()), "Authorization": f"Basic {gigachat_auth_key}"}
    return requests.post(gigachat_auth_url, headers=auth_headers, data={"scope": gigachat_scope}, timeout=60, verify=False).json().get("access_token")

def make_gigachat_headers(access_token):
    return {"Accept": "application/json", "Content-Type": "application/json", "Authorization": f"Bearer {access_token}"}

def call_model(provider, model_name, messages):
    payload_key = "max_tokens" if provider == "gigachat" else "max_completion_tokens"
    payload = {"model": model_name, "messages": messages, "temperature": temperature, payload_key: max_tokens}
    url = gigachat_chat_url if provider == "gigachat" else openrouter_url
    headers = headers_gigachat if provider == "gigachat" else headers_openrouter
    request_kwargs = {"headers": headers, "json": payload, "timeout": (30, 180)}
    request_kwargs.update({"verify": False} if provider == "gigachat" else {})
    return requests.post(url, **request_kwargs)

essays_file = "train_full.xlsx"
predictions_file = "model_scores.xlsx"

gold_df = pd.read_excel(essays_file).copy()
gold_df["essay_id"] = range(1, len(gold_df) + 1)

provider = "gigachat" if model_name in gigachat_models else "openrouter"

if provider == "gigachat":
    gigachat_access_token = get_gigachat_token()
    headers_gigachat = make_gigachat_headers(gigachat_access_token)
    print("GigaChat token refreshed before run.")

rows = []
print("MODEL:", model_name)

for prompt_id, system_prompt in prompt_versions:
    print("PROMPT:", prompt_id)
    for _, row in gold_df.iterrows():
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": build_user_message(row["Essay"], row["Task"])}]
        started_at = time.perf_counter()
        response = call_model(provider, model_name, messages)
        if provider == "gigachat" and response.status_code in [401, 403]:
            gigachat_access_token = get_gigachat_token()
            headers_gigachat = make_gigachat_headers(gigachat_access_token)
            time.sleep(5)
            response = call_model(provider, model_name, messages)
            if response.status_code == 429:
                print("Rate limit, ждём 10 секунд...")
                time.sleep(20)
                response = call_model(provider, model_name, messages)
        model_answer = response.json()["choices"][0]["message"]["content"] if response.ok else f"HTTP {response.status_code}: {response.text}"
        if not rows: print("Первый ответ модели:\n", model_answer)
        model_score = extract_overall_score(model_answer)
        rows.append({"model": model_name, "provider": provider, "prompt_id": prompt_id, "essay_id": row["essay_id"], "task": row["Task"], "essay": row["Essay"], "реальная_оценка": row["Grade"], "комментарии_эталон": row["Comment"], "оценка_модели": model_score, "комментарии_модели": model_answer, "parse_ok": int(pd.notna(model_score)), "elapsed_sec": time.perf_counter() - started_at})

new_predictions_df = pd.DataFrame(rows)
old_predictions_df = pd.read_excel(predictions_file) if os.path.exists(predictions_file) else pd.DataFrame()
predictions_all_df = pd.concat([old_predictions_df, new_predictions_df], ignore_index=True).drop_duplicates(subset=["model", "prompt_id", "essay_id"], keep="last")
predictions_all_df.to_excel(predictions_file, index=False)

print("Сохранено в", predictions_file)
print("Добавлено строк:", len(new_predictions_df))
print("Всего строк после объединения:", len(predictions_all_df))
predictions_all_df


MODEL: nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free
PROMPT: prompt_1
Первый ответ модели:
 10  
The essay fully addresses the task, presents a clear thesis, develops two well‑supported arguments, and concludes appropriately. Paragraphing and linking are effective, vocabulary is wide and accurate, grammar is varied and error‑free, and the register is appropriately formal. No significant language or structural errors were found.
PROMPT: prompt_2
PROMPT: prompt_3
PROMPT: prompt_4
PROMPT: prompt_5
Сохранено в model_scores.xlsx
Добавлено строк: 200
Всего строк после объединения: 1000


,model,provider,prompt_id,essay_id,task,essay,реальная_оценка,комментарии_эталон,оценка_модели,комментарии_модели,parse_ok,elapsed_sec
0,GigaChat-2,gigachat,prompt_1,1,Some say that because some people are living l...,Some people argue that since life expectancy i...,10.0,The essay fully addresses the task and present...,10.0,(И) Оценка: 10\n\nКомментарии: \nВы продемонст...,1,2.805942
1,GigaChat-2,gigachat,prompt_1,2,Some say that because some people are living l...,It is believed that since some part of the pop...,8.0,"The essay presents, extends and supports main ...",7.0,(И) Оценка: 7\n\nКомментарии: \n1. Task Respon...,1,3.041451
2,GigaChat-2,gigachat,prompt_1,3,Some say that because some people are living l...,It is widely argued that the retirement age sh...,8.5,The essay presents fully-developed arguments a...,8.0,(И) Оценка: 8\n\n(Ком) Эссе в целом хорошо стр...,1,2.688268
3,GigaChat-2,gigachat,prompt_1,4,Some say that because some people are living l...,It is argued that since the average life durat...,8.0,"The essay addresses all parts of the task, pre...",2.0,(И) Оценка: 2\n\nКомментарии: \n1. Task Respon...,1,3.274300
4,GigaChat-2,gigachat,prompt_1,5,Some say that because some people are living l...,"In recent years, the issue of raising the reti...",6.0,This essay addresses the topic but lacks a cle...,7.0,(И) Оценка: 7\n\nКомментарии: \n1. Task Respon...,1,2.908472
...,...,...,...,...,...,...,...,...,...,...,...,...
995,nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:...,openrouter,prompt_5,36,"In many countries, people are living longer th...",It goes without saying that due to crucial cha...,10,Comment:\nIt goes without saying that due to c...,8.0,**Оценка: 8**\n\n**Комментарий:** \nЭссе полн...,1,5.855835
996,nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:...,openrouter,prompt_5,37,Some people believe that children should focus...,Some people argue that children should focus m...,9,Some people argue that children should focus m...,7.5,**Оценка:** 7.5/10 \n\n**Комментарий:** \nTh...,1,3.758309
997,nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:...,openrouter,prompt_5,38,Computer technology is rapidly developing and ...,"To begin, over time digital mechanisms will pe...",8,"To begin, over time digital mechanisms will pe...",5.0,**Оценка: 5/10**\n\n**Комментарий:** \nThe es...,1,6.789282
998,nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:...,openrouter,prompt_5,39,Some people believe that schools should mainly...,It is argued that schools should primarily foc...,8,Here's a revised version of your essay with co...,7.0,Оценка: 7 \n\nКомментарий: Эссе полностью отв...,1,5.315666


In [91]:
model_name = "nvidia/nemotron-3-nano-30b-a3b:free"
provider = "openrouter"
essays_file = "train_full.xlsx"
predictions_file = f"answers_{model_name.replace('/', '_').replace(':', '_')}.xlsx"
max_answer_retries = 5
min_answer_len = 20
sleep_between_retries = 5
sleep_between_requests = 3
gold_df = pd.read_excel(essays_file).copy()
gold_df["essay_id"] = range(1, len(gold_df) + 1)
rows = []
print("MODEL:", model_name)

for prompt_id, system_prompt in prompt_versions:
    print("PROMPT:", prompt_id)
    for _, row in gold_df.iterrows():
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": build_user_message(row["Essay"], row["Task"])},
        ]
        started_at = time.perf_counter()
        model_answer = ""
        answer_ok = False
        for attempt in range(1, max_answer_retries + 1):
            time.sleep(sleep_between_requests)
            response = call_model(provider, model_name, messages)
            if response.status_code == 429:
                print("Rate limit, ждём 20 секунд...")
                time.sleep(20)
                response = call_model(provider, model_name, messages)
            model_answer = (response.json()["choices"][0]["message"].get("content", "") if response.ok else f"HTTP {response.status_code}: {response.text}")

            model_answer_clean = str(model_answer).strip()

            if not rows and attempt == 1:
                print("Первый ответ модели:\n", model_answer_clean)

            if len(model_answer_clean) >= min_answer_len:
                answer_ok = True
                break

            print(f"essay_id={row['essay_id']} | {prompt_id} | attempt {attempt}/{max_answer_retries}: ответ пустой или короче {min_answer_len} символов")

            if attempt < max_answer_retries:
                time.sleep(sleep_between_retries)

        model_score = extract_overall_score(model_answer_clean)

        rows.append({
            "model": model_name,
            "provider": provider,
            "prompt_id": prompt_id,
            "essay_id": row["essay_id"],
            "task": row["Task"],
            "essay": row["Essay"],
            "реальная_оценка": row["Grade"],
            "комментарии_эталон": row["Comment"],
            "оценка_модели": model_score,
            "комментарии_модели": model_answer_clean,
            "answer_ok": int(answer_ok),
            "attempts": attempt,
            "elapsed_sec": time.perf_counter() - started_at,
        })

predictions_df = pd.DataFrame(rows)
predictions_df.to_excel(predictions_file, index=False)

print("Сохранено в", predictions_file)
print("Строк:", len(predictions_df))
print("Нормальных ответов:", predictions_df["answer_ok"].sum())
print("Плохих ответов:", len(predictions_df) - predictions_df["answer_ok"].sum())

predictions_df

MODEL: nvidia/nemotron-3-nano-30b-a3b:free
PROMPT: prompt_1
Первый ответ модели:
 None
essay_id=1 | prompt_1 | attempt 1/5: ответ пустой или короче 20 символов
essay_id=5 | prompt_1 | attempt 1/5: ответ пустой или короче 20 символов
essay_id=5 | prompt_1 | attempt 2/5: ответ пустой или короче 20 символов
essay_id=5 | prompt_1 | attempt 3/5: ответ пустой или короче 20 символов
essay_id=5 | prompt_1 | attempt 4/5: ответ пустой или короче 20 символов
essay_id=5 | prompt_1 | attempt 5/5: ответ пустой или короче 20 символов
essay_id=8 | prompt_1 | attempt 1/5: ответ пустой или короче 20 символов
essay_id=9 | prompt_1 | attempt 1/5: ответ пустой или короче 20 символов
essay_id=12 | prompt_1 | attempt 1/5: ответ пустой или короче 20 символов
essay_id=12 | prompt_1 | attempt 2/5: ответ пустой или короче 20 символов
essay_id=15 | prompt_1 | attempt 1/5: ответ пустой или короче 20 символов
essay_id=15 | prompt_1 | attempt 2/5: ответ пустой или короче 20 символов
essay_id=17 | prompt_1 | attempt 

,model,provider,prompt_id,essay_id,task,essay,реальная_оценка,комментарии_эталон,оценка_модели,комментарии_модели,answer_ok,attempts,elapsed_sec
0,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_1,1,Some say that because some people are living l...,Some people argue that since life expectancy i...,10,The essay fully addresses the task and present...,10.0,**Оценка: 10**\n\n**Комментарий:** \n- В ввод...,1,2,23.815609
1,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_1,2,Some say that because some people are living l...,It is believed that since some part of the pop...,8,"The essay presents, extends and supports main ...",NaN,3 \n\nВводное предложение лишь частично переф...,1,1,8.815045
2,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_1,3,Some say that because some people are living l...,It is widely argued that the retirement age sh...,8.5,The essay presents fully-developed arguments a...,NaN,10 \n\nThe essay containsseveral lexical and ...,1,1,8.811833
3,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_1,4,Some say that because some people are living l...,It is argued that since the average life durat...,8,"The essay addresses all parts of the task, pre...",5.0,Оценка: 5 \n\nВводный абзац содержит лишь час...,1,1,9.183950
4,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_1,5,Some say that because some people are living l...,"In recent years, the issue of raising the reti...",6,This essay addresses the topic but lacks a cle...,NaN,None,0,5,70.399445
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_5,36,"In many countries, people are living longer th...",It goes without saying that due to crucial cha...,10,Comment:\nIt goes without saying that due to c...,8.0,**Оценка: 8 из 10**\n\n**Комментарий:** \nЭсс...,1,1,9.759813
196,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_5,37,Some people believe that children should focus...,Some people argue that children should focus m...,9,Some people argue that children should focus m...,6.5,**Оценка:****6.5 / 10**\n\n**Комментарий:** \...,1,1,9.066376
197,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_5,38,Computer technology is rapidly developing and ...,"To begin, over time digital mechanisms will pe...",8,"To begin, over time digital mechanisms will pe...",8.0,**Оценка:** **8 из 10**\n\n**Комментарий:** \...,1,1,9.446788
198,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_5,39,Some people believe that schools should mainly...,It is argued that schools should primarily foc...,8,Here's a revised version of your essay with co...,6.5,**Оценка:** **6.5 / 10**\n\n**Комментарий:** ...,1,1,8.626477


In [93]:
predictions_df.to_excel("aaa.xlsx", index=False)

In [98]:
model_name = "nvidia/nemotron-3-nano-30b-a3b:free"
provider = "openrouter"

predictions_file = "aaa.xlsx"

max_answer_retries = 20
min_answer_len = 20
sleep_between_requests = 3
sleep_between_retries = 5

predictions_df = pd.read_excel(predictions_file)

prompt_dict = dict(prompt_versions)

bad_mask = predictions_df["answer_ok"].fillna(0).astype(int).eq(0)
bad_rows = predictions_df[bad_mask].copy()

print("Будем перепрогонять строк:", len(bad_rows))

for idx, old_row in bad_rows.iterrows():
    prompt_id = old_row["prompt_id"]
    system_prompt = prompt_dict[prompt_id]
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": build_user_message(old_row["essay"], old_row["task"])},]
    started_at = time.perf_counter()
    model_answer_clean = ""
    answer_ok = False
    for attempt in range(1, max_answer_retries + 1):
        time.sleep(sleep_between_requests)
        response = call_model(provider, model_name, messages)
        if response.status_code == 429:
            print("Rate limit, ждём 20 секунд...")
            time.sleep(20)
            response = call_model(provider, model_name, messages)
        if response.ok:
            model_answer = response.json()["choices"][0]["message"].get("content", "")
            model_answer_clean = "" if model_answer is None else str(model_answer).strip()
        else:
            model_answer_clean = f"HTTP {response.status_code}: {response.text}"
        if response.ok and len(model_answer_clean) >= min_answer_len:
            answer_ok = True
            break
        print(f"idx={idx} | essay_id={old_row['essay_id']} | {prompt_id} | attempt {attempt}/{max_answer_retries}: ответ пустой или короче {min_answer_len} символов")
        if attempt < max_answer_retries:
            time.sleep(sleep_between_retries)

    predictions_df.loc[idx, "model"] = model_name
    predictions_df.loc[idx, "provider"] = provider
    predictions_df.loc[idx, "оценка_модели"] = extract_overall_score(model_answer_clean)
    predictions_df.loc[idx, "комментарии_модели"] = model_answer_clean
    predictions_df.loc[idx, "answer_ok"] = int(answer_ok)
    predictions_df.loc[idx, "attempts"] = attempt
    predictions_df.loc[idx, "elapsed_sec"] = time.perf_counter() - started_at

predictions_df.to_excel(predictions_file, index=False)

print("Сохранено в", predictions_file)
print("Перепрогнано строк:", len(bad_rows))
print("Осталось плохих:", predictions_df["answer_ok"].fillna(0).astype(int).eq(0).sum())

predictions_df

Будем перепрогонять строк: 1
idx=30 | essay_id=31 | prompt_1 | attempt 1/20: ответ пустой или короче 20 символов
idx=30 | essay_id=31 | prompt_1 | attempt 2/20: ответ пустой или короче 20 символов
idx=30 | essay_id=31 | prompt_1 | attempt 3/20: ответ пустой или короче 20 символов
idx=30 | essay_id=31 | prompt_1 | attempt 4/20: ответ пустой или короче 20 символов
idx=30 | essay_id=31 | prompt_1 | attempt 5/20: ответ пустой или короче 20 символов
idx=30 | essay_id=31 | prompt_1 | attempt 6/20: ответ пустой или короче 20 символов
idx=30 | essay_id=31 | prompt_1 | attempt 7/20: ответ пустой или короче 20 символов
idx=30 | essay_id=31 | prompt_1 | attempt 8/20: ответ пустой или короче 20 символов
idx=30 | essay_id=31 | prompt_1 | attempt 9/20: ответ пустой или короче 20 символов
idx=30 | essay_id=31 | prompt_1 | attempt 10/20: ответ пустой или короче 20 символов
idx=30 | essay_id=31 | prompt_1 | attempt 11/20: ответ пустой или короче 20 символов
idx=30 | essay_id=31 | prompt_1 | attempt 12/

,model,provider,prompt_id,essay_id,task,essay,реальная_оценка,комментарии_эталон,оценка_модели,комментарии_модели,answer_ok,attempts,elapsed_sec
0,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_1,1,Some say that because some people are living l...,Some people argue that since life expectancy i...,10,The essay fully addresses the task and present...,10.0,**Оценка: 10**\n\n**Комментарий:** \n- В ввод...,1,2,23.815609
1,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_1,2,Some say that because some people are living l...,It is believed that since some part of the pop...,8,"The essay presents, extends and supports main ...",NaN,3 \n\nВводное предложение лишь частично переф...,1,1,8.815045
2,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_1,3,Some say that because some people are living l...,It is widely argued that the retirement age sh...,8.5,The essay presents fully-developed arguments a...,NaN,10 \n\nThe essay containsseveral lexical and ...,1,1,8.811833
3,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_1,4,Some say that because some people are living l...,It is argued that since the average life durat...,8,"The essay addresses all parts of the task, pre...",5.0,Оценка: 5 \n\nВводный абзац содержит лишь час...,1,1,9.183950
4,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_1,5,Some say that because some people are living l...,"In recent years, the issue of raising the reti...",6,This essay addresses the topic but lacks a cle...,7.0,Оценка: 7Комментарий: \n- Введение лишь части...,1,5,71.835449
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_5,36,"In many countries, people are living longer th...",It goes without saying that due to crucial cha...,10,Comment:\nIt goes without saying that due to c...,8.0,**Оценка: 8 из 10**\n\n**Комментарий:** \nЭсс...,1,1,9.759813
196,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_5,37,Some people believe that children should focus...,Some people argue that children should focus m...,9,Some people argue that children should focus m...,6.5,**Оценка:****6.5 / 10**\n\n**Комментарий:** \...,1,1,9.066376
197,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_5,38,Computer technology is rapidly developing and ...,"To begin, over time digital mechanisms will pe...",8,"To begin, over time digital mechanisms will pe...",8.0,**Оценка:** **8 из 10**\n\n**Комментарий:** \...,1,1,9.446788
198,nvidia/nemotron-3-nano-30b-a3b:free,openrouter,prompt_5,39,Some people believe that schools should mainly...,It is argued that schools should primarily foc...,8,Here's a revised version of your essay with co...,6.5,**Оценка:** **6.5 / 10**\n\n**Комментарий:** ...,1,1,8.626477


In [103]:
input_file = "model_scores.xlsx"
output_file = "metrics_summary.xlsx"

df = pd.read_excel(input_file)
df["реальная_оценка"] = pd.to_numeric(df["реальная_оценка"], errors="coerce")
df["оценка_модели"] = pd.to_numeric(df["оценка_модели"], errors="coerce")
df = df.dropna(subset=["реальная_оценка", "оценка_модели"]).copy()

summary_rows = []

for (model, provider, prompt_id), group in df.groupby(["model", "provider", "prompt_id"], dropna=False):
    y_true = group["реальная_оценка"]
    y_pred = group["оценка_модели"]
    error = y_pred - y_true
    abs_error = error.abs()
    sq_error = error ** 2
    summary_rows.append({
        "model": model,
        "provider": provider,
        "prompt_id": prompt_id,
        "n_essays": len(group),
    
        "MAE": abs_error.mean(),
        "RMSE": np.sqrt(sq_error.mean()),
        "tail_ratio": tail_ratio(y_true, y_pred),
    
        "MAPE": mape(y_true, y_pred),
        "SMAPE": smape(y_true, y_pred),
    
        "Bias": error.mean(),
        "abs_Bias": abs_bias(y_true, y_pred),
    
        "Within_1": (abs_error <= 1).mean(),
        "Within_2": (abs_error <= 2).mean(),
    
        "Spearman": spearman_corr(y_true, y_pred),
        "QWK": quadratic_weighted_kappa(y_true, y_pred),
    
        "avg_real_score": y_true.mean(),
        "avg_model_score": y_pred.mean(),
        "avg_elapsed_sec": group["elapsed_sec"].mean() if "elapsed_sec" in group.columns else np.nan
    })
metrics_df = pd.DataFrame(summary_rows).sort_values(["model", "prompt_id"], ascending=[True, True]).reset_index(drop=True)
metrics_df.to_excel(output_file, index=False, sheet_name="metrics")

print("Файл сохранён:", output_file)
metrics_df


Файл сохранён: metrics_summary.xlsx


,model,provider,prompt_id,n_essays,MAE,RMSE,tail_ratio,MAPE,SMAPE,Bias,abs_Bias,Within_1,Within_2,Spearman,QWK,avg_real_score,avg_model_score,avg_elapsed_sec
0,GigaChat-2,gigachat,prompt_1,40,1.10000,1.677051,1.524592,15.436020,17.964784,-0.85000,0.85000,0.775,0.900,0.533830,0.367545,7.3625,6.51250,3.445603
1,GigaChat-2,gigachat,prompt_2,40,4.73750,4.908538,1.036103,63.166889,93.757882,-4.73750,4.73750,0.025,0.050,0.250748,0.011536,7.3625,2.62500,4.253778
2,GigaChat-2,gigachat,prompt_3,40,1.16250,1.708435,1.469621,16.417171,18.845070,-0.91250,0.91250,0.725,0.875,0.527733,0.365333,7.3625,6.45000,3.102315
3,GigaChat-2,gigachat,prompt_4,40,4.53750,4.703058,1.036487,60.435558,87.703218,-4.53750,4.53750,0.025,0.050,0.352022,0.014767,7.3625,2.82500,4.188993
4,GigaChat-2,gigachat,prompt_5,40,1.13750,1.807968,1.589423,17.769353,18.839944,-0.33750,0.33750,0.825,0.900,0.385134,0.132530,7.3625,7.02500,3.780933
5,GigaChat-2-Max,gigachat,prompt_1,40,0.90000,1.346291,1.495879,11.916826,13.655554,-0.75000,0.75000,0.800,0.925,0.542705,0.535322,7.3625,6.61250,8.188539
6,GigaChat-2-Max,gigachat,prompt_2,40,4.43750,4.711024,1.061639,59.162110,87.474443,-4.33750,4.33750,0.100,0.100,0.159939,0.036092,7.3625,3.02500,10.103799
7,GigaChat-2-Max,gigachat,prompt_3,40,0.78750,1.180572,1.499139,10.604671,11.660709,-0.51250,0.51250,0.850,0.950,0.601896,0.573034,7.3625,6.85000,6.368745
8,GigaChat-2-Max,gigachat,prompt_4,40,4.73750,4.903443,1.035027,63.383340,94.581670,-4.73750,4.73750,0.025,0.025,0.202293,0.017616,7.3625,2.62500,11.887381
9,GigaChat-2-Max,gigachat,prompt_5,40,1.12500,1.874166,1.665926,16.069423,19.354845,-0.62500,0.62500,0.700,0.850,0.394590,0.303704,7.3625,6.73750,8.688801
